### RAG Pipelines - Data ingestion to Vector DB Pipeline


In [49]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [50]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try: 
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")
        except Exception as e:
            print(f"X Error: {e}")
    print(f"\nTotal documents loaded:  {len(all_documents)}")
    return all_documents  
all_pdf_documents = process_all_pdfs("../data")    
all_pdf_documents          


Found 3 PDF files to process

Processing: logic_puzzles.pdf
 Loaded 14 pages

Processing: intro_to_csps.pdf
 Loaded 17 pages

Processing: scheduling.pdf
 Loaded 12 pages

Total documents loaded:  43


[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-10-02T16:16:24+00:00', 'title': 'Logic Puzzles', 'author': 'Rick Valenzano', 'moddate': '2026-10-02T16:16:24+00:00', 'source': '../data/pdf/logic_puzzles.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_file': 'logic_puzzles.pdf', 'file_type': 'pdf'}, page_content='Logic Puzzles\nProfessor Rick Valenzano\nCPS721: Artificial Intelligence I'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-10-02T16:16:24+00:00', 'title': 'Logic Puzzles', 'author': 'Rick Valenzano', 'moddate': '2026-10-02T16:16:24+00:00', 'source': '../data/pdf/logic_puzzles.pdf', 'total_pages': 14, 'page': 1, 'page_label': '2', 'source_file': 'logic_puzzles.pdf', 'file_type': 'pdf'}, page_content='Learning Objectives\n2\nStudents should be able to\n1. Encode logic puzzles using Prolog\n2. Use hidden variables to better capture constraints'),
 Document(metadata={'producer': 'PyPDF', 'creator

In [51]:
## Text splitting get into chunks here 
def split_documents(documents, chunk_size = 1000, chunk_overlap=200):
    "split documents into smaller chunks for better RAG performance"
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #to show example of a chunk
    if split_docs:
        print(f"\n Example chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs    
chunks=split_documents(all_pdf_documents)
chunks

Split 43 documents into 43 chunks

 Example chunk:
Content: Logic Puzzles
Professor Rick Valenzano
CPS721: Artificial Intelligence I...
Metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-10-02T16:16:24+00:00', 'title': 'Logic Puzzles', 'author': 'Rick Valenzano', 'moddate': '2026-10-02T16:16:24+00:00', 'source': '../data/pdf/logic_puzzles.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_file': 'logic_puzzles.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-10-02T16:16:24+00:00', 'title': 'Logic Puzzles', 'author': 'Rick Valenzano', 'moddate': '2026-10-02T16:16:24+00:00', 'source': '../data/pdf/logic_puzzles.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_file': 'logic_puzzles.pdf', 'file_type': 'pdf'}, page_content='Logic Puzzles\nProfessor Rick Valenzano\nCPS721: Artificial Intelligence I'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-10-02T16:16:24+00:00', 'title': 'Logic Puzzles', 'author': 'Rick Valenzano', 'moddate': '2026-10-02T16:16:24+00:00', 'source': '../data/pdf/logic_puzzles.pdf', 'total_pages': 14, 'page': 1, 'page_label': '2', 'source_file': 'logic_puzzles.pdf', 'file_type': 'pdf'}, page_content='Learning Objectives\n2\nStudents should be able to\n1. Encode logic puzzles using Prolog\n2. Use hidden variables to better capture constraints'),
 Document(metadata={'producer': 'PyPDF', 'creator

## Embedding And VectorStoreDB

In [52]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [53]:
class EmbeddingsManager:
    "Handles document embedding generation using StenceTransformer"
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """ 
        Initialize the embedding manager
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name 
        self.model = None
        self._load_model()
    def _load_model(self):
        """ Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension}")   
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}") 
            raise    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """ 
        Generate embeddings for a list of texts
        Args: 
           texts: List of text strings to embed
        Returns:
           numpy array of embeddings with shape (len(texts), embedding_dim)   
        """    
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    ## initialise the embedding manager 
embedding_manager=EmbeddingsManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8603.44it/s]


Model Loaded successfully. Embedding dimension: <bound method SentenceTransformer.get_sentence_embedding_dimension of SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)>


## VectorStore


In [54]:
import os
import uuid
import numpy as np
import chromadb
from typing import List, Any
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 86


In [55]:
chunks

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-10-02T16:16:24+00:00', 'title': 'Logic Puzzles', 'author': 'Rick Valenzano', 'moddate': '2026-10-02T16:16:24+00:00', 'source': '../data/pdf/logic_puzzles.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_file': 'logic_puzzles.pdf', 'file_type': 'pdf'}, page_content='Logic Puzzles\nProfessor Rick Valenzano\nCPS721: Artificial Intelligence I'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-10-02T16:16:24+00:00', 'title': 'Logic Puzzles', 'author': 'Rick Valenzano', 'moddate': '2026-10-02T16:16:24+00:00', 'source': '../data/pdf/logic_puzzles.pdf', 'total_pages': 14, 'page': 1, 'page_label': '2', 'source_file': 'logic_puzzles.pdf', 'file_type': 'pdf'}, page_content='Learning Objectives\n2\nStudents should be able to\n1. Encode logic puzzles using Prolog\n2. Use hidden variables to better capture constraints'),
 Document(metadata={'producer': 'PyPDF', 'creator

In [56]:
### Convert the text to embeddings
texts= [doc.page_content for doc in chunks]
texts

### Generate the embedding
embeddings=embedding_manager.generate_embeddings(texts)

## Store in the vectore database
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 43 texts...


Batches: 100%|██████████| 2/2 [00:00<00:00,  8.56it/s]

Generated embeddings with shape: (43, 384)
Adding 43 documents to vector store...
Successfully added 43 documents to vector store
Total documents in collection: 129


### Retriever Pipeline From VectorStore

In [57]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingsManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)
rag_retriever

In [58]:
rag_retriever.retrieve("what is prolog?")

Retrieving documents for query: 'what is prolog?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 43.09it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_247640d4_16',
  'content': 'Prolog Output\n3\n• Two important predicates for output\nwrite(“Some String”) or write(Term)\n• Writes the string or term to output\nnl\n• Writes a newline to output',
  'metadata': {'content_length': 159,
   'creationdate': '2026-09-28T02:34:55+00:00',
   'file_type': 'pdf',
   'total_pages': 17,
   'source': '../data/pdf/intro_to_csps.pdf',
   'creator': 'PyPDF',
   'moddate': '2026-09-28T02:34:55+00:00',
   'source_file': 'intro_to_csps.pdf',
   'doc_index': 16,
   'author': 'Rick Valenzano',
   'page': 2,
   'title': 'Terms',
   'producer': 'PyPDF',
   'page_label': '3'},
  'similarity_score': 0.4289557933807373,
  'distance': 0.5710442066192627,
  'rank': 1},
 {'id': 'doc_a6def9bd_16',
  'content': 'Prolog Output\n3\n• Two important predicates for output\nwrite(“Some String”) or write(Term)\n• Writes the string or term to output\nnl\n• Writes a newline to output',
  'metadata': {'moddate': '2026-09-28T02:34:55+00:00',
   'source': '../data/

In [59]:
rag_retriever.retrieve("scheduling")

Retrieving documents for query: 'scheduling'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 55.70it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_c431dfe7_31',
  'content': 'Scheduling in Prolog\nProfessor Rick Valenzano\nCPS721: Artificial Intelligence I',
  'metadata': {'creationdate': '2026-10-02T16:12:53+00:00',
   'source_file': 'scheduling.pdf',
   'moddate': '2026-10-02T16:12:53+00:00',
   'page_label': '1',
   'producer': 'PyPDF',
   'doc_index': 31,
   'page': 0,
   'source': '../data/pdf/scheduling.pdf',
   'total_pages': 12,
   'title': 'Scheduling in Prolog',
   'file_type': 'pdf',
   'content_length': 79,
   'author': 'Rick Valenzano',
   'creator': 'PyPDF'},
  'similarity_score': 0.26078253984451294,
  'distance': 0.7392174601554871,
  'rank': 1},
 {'id': 'doc_ec94c944_31',
  'content': 'Scheduling in Prolog\nProfessor Rick Valenzano\nCPS721: Artificial Intelligence I',
  'metadata': {'producer': 'PyPDF',
   'total_pages': 12,
   'source': '../data/pdf/scheduling.pdf',
   'page': 0,
   'author': 'Rick Valenzano',
   'creationdate': '2026-10-02T16:12:53+00:00',
   'doc_index': 31,
   'creator': 'PyPDF',

### Integration Vectordb Context pipeline

In [65]:
### Simple RAG pipeline with OpenAI LLM
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
load_dotenv() # read the api key from .env
from openai import OpenAI
print([m.id for m in OpenAI().models.list()])


## initialise the ChatGPT LLM (we need to set up the openAI key)

llm = ChatOpenAI(
    model="gpt-5-mini",   # see below
    temperature=0.1,
    max_tokens=1024,
)
## 2. Simple RAG function: retrieve context + generate response

def rag_simple(query, retriever,llm,top_k=3):
    ## retrive the context 
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results] ) if results else ""
    if not context:
        return "No relevant context found to answer the question"
    ##generate the answer using openai LLM
    prompt=f""" Use the following cntext to  answer the question concisely
        context:
        {context}

        Question: {query}
        Answer:"""
    
    response=llm.invoke([prompt.format(context = context, query = query)])
    return response.content



['gpt-5.4-2026-03-05', 'gpt-5-nano', 'gpt-5.4-mini', 'tts-1', 'gpt-5-mini', 'gpt-5-mini-2025-08-07', 'gpt-realtime-2.1-mini', 'gpt-realtime-2025-08-28', 'omni-moderation-2024-09-26', 'gpt-live-transcribe', 'gpt-5.6-luna', 'omni-moderation-latest', 'gpt-realtime-2', 'gpt-5-search-api', 'gpt-5.5-2026-04-23', 'gpt-5.5-pro', 'gpt-4o-mini-transcribe-2025-03-20', 'gpt-audio-2025-08-28', 'gpt-5.4-mini-2026-03-17', 'gpt-4o-2024-11-20', 'gpt-5.1', 'whisper-1', 'gpt-5-pro', 'gpt-realtime-whisper', 'gpt-4o-mini-transcribe-2025-12-15', 'gpt-5-pro-2025-10-06', 'gpt-5.6-sol', 'gpt-realtime', 'tts-1-hd', 'gpt-realtime-1.5', 'gpt-5.2-pro', 'text-embedding-3-large', 'chat-latest', 'gpt-5.4-pro-2026-03-05', 'gpt-realtime-translate', 'gpt-image-1.5', 'gpt-audio-1.5', 'gpt-5.4-pro', 'gpt-realtime-mini-2025-12-15', 'gpt-4o-mini-tts', 'gpt-4.1-mini', 'gpt-4o-mini', 'gpt-4o-mini-tts-2025-12-15', 'gpt-audio', 'gpt-5.3-codex', 'gpt-5-search-api-2025-10-14', 'gpt-5-2025-08-07', 'tts-1-1106', 'gpt-5.2-pro-2025-1

In [66]:
answer=rag_simple("What is Prolog?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is Prolog?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:05<00:00,  5.57s/it]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Prolog is a declarative logic programming language that represents knowledge as facts and rules and answers queries by logical inference. It includes predicates for I/O such as write(Term) to print and nl to print a newline.


## Enhanced RAG Pipeline Features


In [68]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("how to code in Prolog?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'how to code in Prolog?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.77it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Answer: Quick overview — a Prolog program is a set of facts and rules. Sentences end with a period. Variables start with an uppercase letter; atoms/predicate names start lowercase (or are quoted). Use % for single-line and /*...*/ for block comments. Load a file with consult('file.pl'). or [file].

Basic elements
- Fact: parent(john,mary).
- Rule: grandparent(X,Z) :- parent(X,Y), parent(Y,Z).
- Query (interactive): ?- grandparent(john,sue).
- Lists: [Head|Tail], empty list [].
- Arithmetic: use is and arithmetic operators, compare with =:=, >, <, etc.

Output
- write(Term). prints Term.
- nl. prints newline.
Example:
parent(john,mary).
parent(mary,sue).

grandparent(X,Z) :-
    parent(X,Y),
    parent(Y,Z).

greet(Name) :-
    write('Hello, '), write(Name), nl.

Usage in the toplevel:
?- grandparent(john,sue).
true.

?- greet('Alice').
Hello, Alice
true.

Small recursive example (sum 1..N):
sum(0,0).
sum(